In [ ]:
# API KEY를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API KEY 정보로드
load_dotenv()

In [ ]:
from langchain_teddynote import logging

# 프로젝트 이름을 입력합니다.
logging.langsmith("CH01")

In [ ]:
import json
from typing import Any, Dict, List, Optional

from langchain.agents import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI
from langchain.agents import create_tool_calling_agent, AgentExecutor

@tool
def query_analysis_tool(
    user_question: str,
    ambiguity_assessment: Dict[str, Any],
    query_candidates: List[Dict[str, Any]],
    final_query: Optional[Dict[str, Any]] = None,
    extracted_keywords: Optional[Dict[str, Any]] = None,
    questions_to_user: Optional[List[str]] = None,
    iteration: int = 0,
) -> Dict[str, Any]:
    """
    Query Analysis Agent가 생성한 분석 결과를 표준 포맷으로 반환한다.

    - ambiguity_assessment: 요소별 모호성 판단 결과
    - query_candidates: 검색용 쿼리 후보들
    - final_query: (확정 시) 최종 쿼리
    - extracted_keywords: (확정 시) 핵심 키워드(동적 개수)
    - questions_to_user: (모호 시) 사용자에게 물어볼 질문들
    - iteration: 현재 반복 횟수
    """
    return {
        "iteration": iteration,
        "input_question": user_question,
        "ambiguity_assessment": ambiguity_assessment,
        "query_candidates": query_candidates,
        "final_query": final_query,
        "extracted_keywords": extracted_keywords,
        "questions_to_user": questions_to_user or [],
        "need_user_input": bool(questions_to_user) and (final_query is None),
    }


tools = [query_analysis_tool]

gpt = ChatOpenAI(model="gpt-5o")

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are Query Analysis Agent for research gap literature search.

Your job:
- Analyze user's question for ambiguity across dimensions:
  (research_domain, target_data, methodology, time_range, task_goal, constraints)
- If ambiguous, propose:
  - 2~5 query_candidates suitable for paper search
  - questions_to_user that will reduce ambiguity
  - include strategies used (synonym expansion, method keyword injection, filters, negative keywords)
- If user has provided feedback or answered questions, incorporate it and refine again.
- Stop when ambiguity is sufficiently resolved OR when max iterations are reached.
- When resolved, output final_query and extracted_keywords (dynamic size).
- ALWAYS call the tool `query_analysis_tool` with a complete structured payload.
- Do NOT output raw text analysis as final; instead, use the tool to emit the result object.

Important formatting requirements for tool call args:
- ambiguity_assessment must be a dict with:
  - dimensions: dict for each dimension with {specified: bool, specificity: "low|medium|high", notes: str}
  - overall_ambiguous: bool
  - rationale: str
- query_candidates must be list of dicts each with:
  - query: str
  - applied_strategies: list[str]
  - filters: dict
  - negative_keywords: list[str]
  - why_better: str
- final_query (if resolved) must be dict with:
  - query: str
  - filters: dict
  - negative_keywords: list[str]
  - assumptions: list[str]
- extracted_keywords (if resolved) must be dict with:
  - keywords: list[str]
  - keyword_groups: dict (domain/data/method/evaluation/application)
  - notes: str
""",
        ),
        (
            "human",
            """
User question:
{input}

(If user feedback exists, it will appear below.)
{user_feedback}
""",
        ),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ]
)

gpt_agent = create_tool_calling_agent(gpt, tools, prompt)

agent_executor = AgentExecutor(
    agent=gpt_agent,
    tools=tools,
    verbose=False,
    max_iterations=10,
    handle_parsing_errors=True,
)


question = ""

user_feedback = ""  # HITL로 누적

final_step = None

for step in agent_executor.iter({"input": question, "user_feedback": user_feedback}):
    final_step = step

    # intermediate_step: tool call 결과 가로채기
    if output := step.get("intermediate_step"):
        action, value = output[0]

        if action.tool == "query_analysis_tool":
            # value는 tool return dict
            print("\n================ TOOL RESULT ================")
            print(json.dumps(value, ensure_ascii=False, indent=2))

            # 모호하면 사용자에게 질문 & 쿼리 후보 보여주고 피드백 받기
            if value.get("need_user_input", False):
                print("\n[Agent 질문]")
                for i, q in enumerate(value.get("questions_to_user", []), 1):
                    print(f"{i}. {q}")

                print("\n[쿼리 후보]")
                for i, c in enumerate(value.get("query_candidates", []), 1):
                    print(f"\n({i}) Query: {c.get('query')}")
                    print(f"    strategies: {c.get('applied_strategies')}")
                    print(f"    filters: {c.get('filters')}")
                    print(f"    negative: {c.get('negative_keywords')}")
                    print(f"    why: {c.get('why_better')}")

                chosen = input("\n원하는 후보 번호를 고르거나(예: 2), 추가 요구사항/답변을 적어줘:\n").strip()
                user_feedback = f"""
[USER FEEDBACK / ANSWERS]
{chosen}

(You must incorporate this feedback and refine queries.)
"""
                continue

            if value.get("final_query") is not None:
                print("\n최종 Query")
                break


if final_step and "output" in final_step:
    print("\n================ FINAL OUTPUT ================")
    print(final_step["output"])
